# 04 · Regularization — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. Laplacian versus damping

Compare at similar misfit, not equal numeric lambda. Laplacian suppresses
curvature and preserves broad amplitude; damping suppresses amplitude directly.

## 2. Nonzero target

A reference model affects only directions seen by $L$ and becomes influential
where data are weak.

## 3–4. Strength criteria and a custom operator

L-curve, ABIC, and CV can disagree because their loss functions differ. A
first-difference matrix is a valid custom operator; inspect its null space
(constant slip) before interpreting it.


In [ ]:
lap = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="laplacian",
    regularization_strength=1.0,
)
damped = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="damping",
    regularization_strength=1.0,
)
target = np.full(fault.n_patches, 0.2)
toward_target = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="damping",
    regularization_strength=10.0, regularization_target=target,
)
first_difference = np.diff(np.eye(fault.n_patches), axis=0)
custom = geodef.solve(
    fault, datasets=gnss, components="dip", regularization=first_difference,
    regularization_strength=1.0,
)
print("roughness lap/damping:", np.std(np.diff(lap.dip_slip)), np.std(np.diff(damped.dip_slip)))
print("target/custom means:", toward_target.dip_slip.mean(), custom.dip_slip.mean())


## Interpretation and alternatives

Operator choice changes which model features count as costly; strength alone never describes the prior.
